# Solutions — Retries, Timeouts, Rate Limits & Circuit Breakers

Exercise notebook: `07_apis/04_retries_rate_limits.ipynb`. Each solution has: working code, an explanation, complexity, common mistakes, and a Java-style vs Pythonic comparison where it teaches something.

### Setup (same as the exercise notebook)

In [ ]:
import sys
from pathlib import Path
LAB = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets").is_dir() and (p / "labkit").is_dir())   # lab root (works from any folder)
sys.path.insert(0, str(LAB))                                # make `labkit` importable from the notebook
from labkit.mock_api import MockSession, BASE_URL
import requests

session = MockSession()                                     # offline fake with the same surface as requests.Session
resp = session.get(f"{BASE_URL}/customers", params={"page": 1, "page_size": 3}, timeout=5)
print(resp.status_code, resp.json()["total"], [c["customer_id"] for c in resp.json()["data"]])
# Prefer real HTTP? In a terminal:  python -m labkit.mock_server --port 8765
# then use requests.Session() with base_url "http://127.0.0.1:8765/v1" — your functions shouldn't change at all.

## Exercise 1 — 🟢 Easy · Classify retryable failures

**Working solution**

In [ ]:
import requests

RETRYABLE_STATUS = frozenset({408, 425, 429, 500, 502, 503, 504})
RETRYABLE_EXC = (requests.Timeout, requests.ConnectionError, TimeoutError, ConnectionError)

def is_retryable(outcome):
    if isinstance(outcome, bool):
        return False
    if isinstance(outcome, int):
        return outcome in RETRYABLE_STATUS
    return isinstance(outcome, RETRYABLE_EXC)

**Explanation**

A `frozenset` of codes + a tuple of exception classes makes the policy explicit data. Retrying a 400/422 just sends the same bad request again.

**Complexity:** O(1).

**Common mistakes**

- Retrying all 4xx.
- Retrying non-idempotent POSTs blindly (duplicates!) — use idempotency keys.

## Exercise 2 — 🟡 Medium · Backoff schedule with optional jitter

**Working solution**

In [ ]:
def backoff_delays(retries, base=0.5, factor=2.0, max_delay=30.0, jitter=None):
    delays = []
    for attempt in range(retries):
        delay = min(base * factor ** attempt, max_delay)
        if jitter is not None:
            delay = jitter(delay)
        delays.append(round(delay, 3))
    return delays

**Explanation**

Jitter spreads out retries from many clients so they don't all hit the recovering server at the same instant (the 'thundering herd').

**Complexity:** O(retries).

**Common mistakes**

- No cap → 2^20 seconds of waiting.

## Exercise 3 — 🟡 Medium · Parse Retry-After

**Working solution**

In [ ]:
from datetime import datetime, timezone
from email.utils import parsedate_to_datetime

def parse_retry_after(value, now=None):
    if value is None:
        return None
    text = str(value).strip()
    if text.isdigit():
        return float(text)
    try:
        when = parsedate_to_datetime(text)
    except (TypeError, ValueError, IndexError):
        return None
    if when is None:
        return None
    now = now or datetime.now(timezone.utc)
    return max(0.0, (when - now).total_seconds())

**Explanation**

The stdlib's `email.utils` parses RFC 2822 dates (used by HTTP). Clamping to ≥ 0 protects against clock skew.

**Complexity:** O(1).

**Common mistakes**

- `int(header)` → crashes on the date form.

## Exercise 4 — 🟠 Hard · GET with retries, backoff and Retry-After

**Working solution**

In [ ]:
import time
import requests

_RETRY_STATUS = {408, 425, 429, 500, 502, 503, 504}

def get_with_retry(session, url, params=None, max_attempts=4, base_delay=0.5, sleep=time.sleep, timeout=10):
    for attempt in range(1, max_attempts + 1):
        get_with_retry.last_attempts = attempt
        try:
            resp = session.get(url, params=params, timeout=timeout)
        except (requests.Timeout, requests.ConnectionError):
            if attempt == max_attempts:
                raise
            sleep(base_delay * 2 ** (attempt - 1))
            continue
        if resp.status_code not in _RETRY_STATUS:
            resp.raise_for_status()
            return resp
        if attempt == max_attempts:
            resp.raise_for_status()
        retry_after = resp.headers.get("Retry-After")
        wait = float(retry_after) if retry_after and retry_after.isdigit() else base_delay * 2 ** (attempt - 1)
        sleep(wait)

get_with_retry.last_attempts = 0

**Explanation**

The loop has three exits: success (return), permanent failure (raise now), exhausted retries (raise last). Server hints (`Retry-After`) beat your own guess.
In production you can also mount `urllib3.util.Retry` on a `requests` adapter — but you should know what it does.

**Complexity:** O(max_attempts) requests.

**Common mistakes**

- Sleeping after the final attempt.
- Ignoring `Retry-After` and getting banned.
- Catching `requests.RequestException` (includes `HTTPError` from bugs).

## Challenge — 🔴 Very Hard · Resilient client: token bucket + retries + circuit breaker

**Working solution**

In [ ]:
import time
import requests

class CircuitOpen(Exception):
    pass

_RETRY = {408, 425, 429, 500, 502, 503, 504}

class ResilientClient:
    def __init__(self, session, base_url, rate_per_sec=10, burst=10, max_attempts=3,
                 failure_threshold=5, cooldown=30, clock=time.monotonic, sleep=time.sleep):
        self.session, self.base_url = session, base_url.rstrip("/")
        self.rate, self.capacity = rate_per_sec, burst
        self.tokens = float(burst)
        self.max_attempts = max_attempts
        self.failure_threshold, self.cooldown = failure_threshold, cooldown
        self.clock, self.sleep = clock, sleep
        self._last_refill = clock()
        self.consecutive_failures = 0
        self.opened_at = None
        self.metrics = {"requests": 0, "retries": 0, "throttled_waits": 0, "circuit_rejections": 0}

    # -- rate limiting ---------------------------------------------------
    def _refill(self):
        now = self.clock()
        self.tokens = min(self.capacity, self.tokens + (now - self._last_refill) * self.rate)
        self._last_refill = now

    def _acquire(self):
        self._refill()
        if self.tokens < 1:
            self.metrics["throttled_waits"] += 1
            self.sleep((1 - self.tokens) / self.rate)
            self._refill()
        self.tokens -= 1

    # -- circuit breaker -------------------------------------------------
    def _check_circuit(self):
        if self.opened_at is None:
            return
        if self.clock() - self.opened_at < self.cooldown:
            self.metrics["circuit_rejections"] += 1
            raise CircuitOpen(f"circuit open for {self.base_url}")
        # cooldown passed: half-open, allow one trial call

    def _record(self, success):
        if success:
            self.consecutive_failures, self.opened_at = 0, None
            return
        self.consecutive_failures += 1
        if self.opened_at is not None or self.consecutive_failures >= self.failure_threshold:
            self.opened_at = self.clock()

    # -- public API ------------------------------------------------------
    def get(self, path, **params):
        self._check_circuit()
        url = f"{self.base_url}/{path.lstrip('/')}"
        try:
            resp = self._attempts(url, params)
        except (requests.Timeout, requests.ConnectionError, requests.HTTPError) as exc:
            status = getattr(getattr(exc, "response", None), "status_code", None)
            self._record(success=status is not None and status not in _RETRY)
            raise
        self._record(success=True)
        return resp.json()

    def _attempts(self, url, params):
        for attempt in range(1, self.max_attempts + 1):
            if attempt > 1:
                self.metrics["retries"] += 1
            self._acquire()
            self.metrics["requests"] += 1
            try:
                resp = self.session.get(url, params=params or None, timeout=10)
            except (requests.Timeout, requests.ConnectionError):
                if attempt == self.max_attempts:
                    raise
                self.sleep(0.5 * 2 ** (attempt - 1))
                continue
            if resp.status_code not in _RETRY:
                resp.raise_for_status()
                return resp
            if attempt == self.max_attempts:
                resp.raise_for_status()
            hint = resp.headers.get("Retry-After")
            self.sleep(float(hint) if hint and hint.isdigit() else 0.5 * 2 ** (attempt - 1))

**Explanation**

Three classic resilience patterns composed in one small class. The token bucket smooths bursts; retries absorb blips; the breaker stops a failing dependency from
consuming your whole time budget (and lets it recover). Note a non-retryable 4xx means *the server is healthy* — it doesn't trip the breaker.

**Complexity:** O(1) per call plus retries.

**Common mistakes**

- Counting each retry attempt as a breaker failure (trips far too early).
- Opening the circuit forever with no half-open trial.